# Bank mijozlari churn bashorati

Bu loyihada bank mijozlarining bankdan ketishini (churn) oldindan bashorat qilamiz. Ketishga moyil mijozlarni vaqtida aniqlab, ularni ushlab qolish choralarini ko'rish mumkin bo'ladi.

**Dataset:** Kaggle, *Churn Modelling* (10 000 mijoz).

| Ustun | Ma'nosi |
|---|---|
| `CreditScore` | kredit balli |
| `Geography` | mamlakat |
| `Gender` | jinsi |
| `Age` | yoshi |
| `Tenure` | bankda necha yildan beri mijoz |
| `Balance` | balans |
| `NumOfProducts` | foydalanadigan mahsulotlar soni |
| `HasCrCard` | kredit kartasi bormi |
| `IsActiveMember` | faol mijozmi |
| `EstimatedSalary` | taxminiy yillik daromad |
| `Exited` | **target**: 1 - bankdan ketgan, 0 - qolgan |

**Ish tartibi:** ma'lumotni o'rganamiz va tozalaymiz, train/test ga bo'lamiz, 5 ta model quramiz (Logistic Regression, SVM, Decision Tree, Random Forest, XGBoost), ularni Accuracy, Precision, Recall va F1 bo'yicha solishtiramiz va xulosa chiqaramiz.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/alishermutalov/praktikum-datasets/refs/heads/praktikum/Churn_Modelling.xls"
df = pd.read_csv(url, index_col='RowNumber')
df.head()

## 1. Ma'lumotni o'rganamiz va tozalaymiz
### 1.1. Bo'sh qiymatlar, dublikatlar va umumiy statistika

In [ ]:
print("Shakl:", df.shape)
print("\nBo'sh qiymatlar:\n", df.isnull().sum())
print("\nDublikatlar soni:", df.duplicated().sum())
df.describe()

**Tushuntirish**
- `df.shape`: qator va ustunlar soni.
- `df.isnull().sum()`: har bir ustundagi bo'sh qiymatlar soni.
- `df.duplicated().sum()`: takroriy qatorlar soni.
- `df.describe()`: min, max, o'rtacha, std kabi statistikalar. Anomaliyalarni (g'alati qiymatlarni) shu yerdan qidiramiz.

### 1.2. Target taqsimoti va anomaliyalar

In [ ]:
print(df['Exited'].value_counts())
print(df['Exited'].value_counts(normalize=True).round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['Age'].plot(kind='box', ax=axes[0], title='Age')
df['CreditScore'].plot(kind='box', ax=axes[1], title='CreditScore')
df['Balance'].plot(kind='box', ax=axes[2], title='Balance')
plt.show()

print("Balansi 0 bo'lgan mijozlar ulushi:", round((df['Balance'] == 0).mean(), 3))

**Tushuntirish**
- `value_counts()`: churn va non-churn soni; `normalize=True` ulushini beradi. Klasslar nomutanosib ekanini shu yerdan ko'ramiz.
- `plot(kind='box')`: boxplot orqali chetki qiymatlarni (outlier) ko'ramiz. `Age` da katta yoshdagi mijozlar bor, lekin bu real qiymatlar (xato emas), shuning uchun o'chirmaymiz.
- `(df['Balance'] == 0).mean()`: balansi nol bo'lgan mijozlar ulushi. Bu anomaliya emas, haqiqiy holat (hisobda pul yo'q), shuning uchun saqlaymiz.

### 1.3. Keraksiz ustunlarni olib tashlaymiz

In [ ]:
df = df.drop(columns=['CustomerId', 'Surname'])
df.head()

**Tushuntirish**
- `CustomerId` va `Surname` identifikatorlar: bashoratga foydali ma'lumot bermaydi, modelni chalg'itishi mumkin.
- `RowNumber` allaqachon indeks sifatida o'qilgan, shuning uchun alohida ustun emas.

### 1.4. Kategorik ustunlarni raqamga aylantiramiz (One-Hot Encoding)

In [ ]:
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True, dtype=int)
df.head()

**Tushuntirish**
- `pd.get_dummies`: matnli ustunlarni 0/1 ustunlarga aylantiradi.
- `columns=['Geography', 'Gender']`: faqat shu ikki kategorik ustun o'zgaradi.
- `drop_first=True`: har ustundan bitta kategoriyani tashlaymiz (dummy trap oldini oladi). Natijada `Geography_Germany`, `Geography_Spain`, `Gender_Male` hosil bo'ladi.
- `dtype=int`: True/False o'rniga 1/0 bo'ladi.

## 2. Ma'lumotni train va test ga bo'lamiz (80% / 20%)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Exited'])
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

**Tushuntirish**
- `X`: belgilar, `y`: target `Exited`.
- `test_size=0.2`: 20% test uchun ajratiladi.
- `random_state=42`: natija har safar bir xil chiqadi.
- `stratify=y`: churn ulushi train va testda bir xil saqlanadi.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

**Tushuntirish**
- `StandardScaler`: belgilarni o'rtacha 0, std 1 ga keltiradi. Logistic Regression va SVM masshtabga sezgir, shuning uchun kerak.
- `fit_transform(X_train)`: parametrlarni faqat train'dan o'rganadi.
- `transform(X_test)`: testga train parametrlari qo'llanadi (data leakage bo'lmasligi uchun test'da `fit` qilinmaydi).
- Daraxtli modellar (Decision Tree, Random Forest, XGBoost) masshtab talab qilmaydi, ularga asl ma'lumotni beramiz.

## 3. Modellarni quramiz

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# (model, masshtablangan ma'lumot kerakmi?)
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'SVM': (SVC(kernel='rbf', random_state=42), True),
    'Decision Tree': (DecisionTreeClassifier(max_depth=5, random_state=42), False),
    'Random Forest': (RandomForestClassifier(n_estimators=200, random_state=42), False),
    'XGBoost': (XGBClassifier(n_estimators=200, learning_rate=0.1,
                              eval_metric='logloss', random_state=42), False),
}

**Tushuntirish**
- Har bir modelga `(model, scaled_kerakmi)` juftligini beramiz: `True` bo'lsa masshtablangan ma'lumot ishlatiladi.
- `LogisticRegression(max_iter=1000)`: konvergensiya uchun yetarli iteratsiya.
- `SVC(kernel='rbf')`: chiziqli bo'lmagan chegarani topa oladigan SVM.
- `DecisionTreeClassifier(max_depth=5)`: chuqurlikni cheklab, overfitting'ni kamaytiramiz.
- `RandomForestClassifier(n_estimators=200)`: 200 ta daraxtdan iborat ansambl.
- `XGBClassifier(learning_rate=0.1)`: gradient boosting; `eval_metric='logloss'` ogohlantirishlarni oldini oladi.

## 4. Modellarni baholaymiz

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []
for name, (model, use_scaled) in models.items():
    Xtr = X_train_sc if use_scaled else X_train
    Xte = X_test_sc if use_scaled else X_test
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred),
        'F1-score': f1_score(y_test, pred),
    })

results_df = pd.DataFrame(results).set_index('Model').round(3)
results_df.sort_values('F1-score', ascending=False)

**Tushuntirish**
- `for name, (model, use_scaled) in models.items()`: barcha modellarni bir xil usulda o'qitamiz va baholaymiz.
- `Xtr`, `Xte`: modelga qarab masshtablangan yoki asl ma'lumot tanlanadi.
- `model.fit(...)`: train ma'lumotida o'qitadi; `model.predict(...)`: test uchun 0/1 bashorat beradi.
- **Accuracy**: umumiy to'g'ri javoblar ulushi.
- **Precision**: "ketadi" deb aytilganlarning nechasi haqiqatan ketgan.
- **Recall**: haqiqatan ketganlarning nechasi topilgan.
- **F1-score**: precision va recall ning garmonik o'rtachasi.

## 5. Natijalarni tahlil qilamiz

In [ ]:
results_df.plot(kind='bar', figsize=(11, 5))
plt.title('Modellar solishtiruvi')
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.legend(loc='lower right')
plt.show()

best_name = results_df['F1-score'].idxmax()
print("F1 bo'yicha eng yaxshi model:", best_name)
print(results_df.loc[best_name])

**Tushuntirish**
- `plot(kind='bar')`: 4 ta metrikani har bir model uchun ustunli diagrammada ko'rsatadi.
- `idxmax()`: F1 eng yuqori bo'lgan model nomini topadi.

In [ ]:
best_model = models[best_name][0]

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()
    imp.plot(kind='barh', figsize=(8, 5), title=f'Belgilar muhimligi ({best_name})')
    plt.show()
    print(imp.sort_values(ascending=False).round(3))

**Tushuntirish**
- `hasattr(..., 'feature_importances_')`: bu atribut faqat daraxtli modellarda bor.
- `pd.Series(..., index=X.columns)`: muhimlik qiymatlarini belgi nomlariga bog'laydi.
- `barh`: qaysi belgi churnga eng ko'p ta'sir qilganini ko'rsatadi.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

Xte_best = X_test_sc if models[best_name][1] else X_test
cm = confusion_matrix(y_test, best_model.predict(Xte_best))
ConfusionMatrixDisplay(cm, display_labels=['Qoldi', 'Ketdi']).plot(cmap='Blues')
plt.title(f'Confusion matrix: {best_name}')
plt.show()

**Tushuntirish**
- `confusion_matrix`: to'g'ri va noto'g'ri bashoratlar jadvali. Pastki chap katak (ketgan, lekin "qoladi" deyilgan) bankka eng qimmatga tushadigan xato.

## 6. Xulosa va tavsiyalar

**Modellarni solishtirish**
- Klasslar nomutanosib (qoladiganlar ancha ko'p), shuning uchun **accuracy aldamchi**: hamma mijozni "qoladi" deb aytgan model ham yuqori accuracy oladi. Shu sababli **Recall** va **F1-score** ga tayanamiz.
- Eng yaxshi model 5-bo'limda F1 bo'yicha aniqlanadi.

**Bank uchun tavsiyalar** (belgilar muhimligi grafigida yuqori chiqqan omillarga qarab)
1. **Yoshi katta mijozlar:** ular orasida churn yuqori bo'lsa, alohida saqlab qolish dasturi va shaxsiy xizmat taklif qilamiz.
2. **Nofaol mijozlar** (`IsActiveMember = 0`): bildirishnomalar, cashback va bonuslar orqali faollashtiramiz.
3. **Mahsulotlar soni:** juda ko'p yoki juda kam mahsulotli mijozlarda churn yuqori bo'lsa, ularga individual taklif tayyorlaymiz.
4. **Geografiya:** biror mamlakatda churn yuqori bo'lsa, o'sha bozorda narx va xizmat sifatini ko'rib chiqamiz.
5. **Balansi yuqori mijozlar:** ketishi bank uchun eng qimmat, shuning uchun ularga ustuvor e'tibor va maxsus shartlar beramiz.
6. Model yordamida churn ehtimoli yuqori mijozlar ro'yxatini oldindan tuzib, ularga maqsadli kampaniya o'tkazamiz.